### data analyzer

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



In [2]:
data = {
    "Date": [
        "2026-01-01", "2026-01-02", "2026-01-03",
        "2026-01-04", "2026-01-05", "2026-01-06",
        "2026-01-07", "2026-01-08", "2026-01-09",
        "2026-01-10", "2026-01-11", "2026-01-12"
    ],

    "Product": [
        "Laptop", "Mouse", "Keyboard", "Mobile",
        "Laptop", "Mouse", "Monitor", "Mobile",
        "Keyboard", "Laptop", "Monitor", "Mouse"
    ],

    "Category": [
        "Electronics", "Accessories", "Accessories",
        "Electronics", "Electronics", "Accessories",
        "Electronics", "Electronics", "Accessories",
        "Electronics", "Electronics", "Accessories"
    ],

    "Price": [
        50000, 500, 1200, 20000,
        50000, 500, 12000, 20000,
        1200, 50000, 12000, 500
    ],

    "Quantity Sold": [
        2, 10, 5, 3, 1, 15, 2, 4, 8, 2, 3, 12
    ]
}

df = pd.DataFrame(data)

df["Total Sales"] = df["Price"] * df["Quantity Sold"]

df.to_csv("retail_sales.csv", index=False)

print("CSV file created successfully")
display(df)

CSV file created successfully


,Date,Product,Category,Price,Quantity Sold,Total Sales
0,2026-01-01,Laptop,Electronics,50000,2,100000
1,2026-01-02,Mouse,Accessories,500,10,5000
2,2026-01-03,Keyboard,Accessories,1200,5,6000
3,2026-01-04,Mobile,Electronics,20000,3,60000
4,2026-01-05,Laptop,Electronics,50000,1,50000
5,2026-01-06,Mouse,Accessories,500,15,7500
6,2026-01-07,Monitor,Electronics,12000,2,24000
7,2026-01-08,Mobile,Electronics,20000,4,80000
8,2026-01-09,Keyboard,Accessories,1200,8,9600
9,2026-01-10,Laptop,Electronics,50000,2,100000


In [3]:
class RetailAnalyzer:

    def __init__(self):
        self.data = None

    # Load CSV file
    def load_data(self, file_path):

        if not file_path.endswith(".csv"):
            print("Please select CSV file")
            return False

        if not os.path.exists(file_path):
            print("File not found")
            return False

        try:
            df = pd.read_csv(file_path)

            columns = [
                "Date", "Product", "Category",
                "Price", "Quantity Sold", "Total Sales"
            ]

            for col in columns:
                if col not in df.columns:
                    print(col, "column is missing")
                    return False

            if df.empty:
                print("Dataset is empty")
                return False

            df["Date"] = pd.to_datetime(
                df["Date"], errors="coerce"
            )

            df["Price"] = pd.to_numeric(
                df["Price"], errors="coerce"
            )

            df["Quantity Sold"] = pd.to_numeric(
                df["Quantity Sold"], errors="coerce"
            )

            df["Product"] = df["Product"].fillna("Unknown")
            df["Category"] = df["Category"].fillna("Unknown")

            df = df.dropna(
                subset=["Date", "Price", "Quantity Sold"]
            )

            df = df[
                (df["Price"] >= 0) &
                (df["Quantity Sold"] >= 0)
            ]

            if df.empty:
                print("No valid data found")
                return False

            df["Total Sales"] = (
                df["Price"] * df["Quantity Sold"]
            )

            self.data = df.reset_index(drop=True)

            print("Data loaded successfully")
            return True

        except Exception as e:
            print("Error:", e)
            return False

    # Calculate sales metrics using NumPy Array
    def calculate_metrics(self):

        if self.data is None:
            print("Load data first")
            return

        sales_array = np.array(self.data["Total Sales"])

        total_sales = np.sum(sales_array)
        average_sales = np.mean(sales_array)

        popular_product = self.data.groupby(
            "Product"
        )["Quantity Sold"].sum().idxmax()

        print("Total Sales:", total_sales)
        print("Average Sales:", round(average_sales, 2))
        print("Most Popular Product:", popular_product)

        daily_sales = self.data.groupby(
            "Date"
        )["Total Sales"].sum().sort_index()

        sales_array = np.array(daily_sales)

        if len(sales_array) > 1 and sales_array[0] != 0:

            growth = (
                (sales_array[-1] - sales_array[0])
                / sales_array[0]
            ) * 100

            print("Growth Percentage:", round(growth, 2), "%")

        else:
            print("Growth cannot be calculated")

    # Aggregate data
    def aggregate_data(self):

        print("\nProduct Wise Sales")

        product_sales = self.data.groupby(
            "Product"
        )["Total Sales"].sum()

        display(product_sales)

        print("\nCategory Wise Sales")

        category_sales = self.data.groupby(
            "Category"
        )["Total Sales"].sum()

        display(category_sales)

        print("\nDate Wise Sales")

        date_sales = self.data.groupby(
            "Date"
        )["Total Sales"].sum()

        display(date_sales)

    # Filter data
    def filter_data(self):

        print("1. Filter by Category")
        print("2. Filter by Date Range")

        choice = input("Enter choice: ")

        if choice == "1":

            category = input("Enter category: ")

            result = self.data[
                self.data["Category"].str.lower()
                == category.lower()
            ]

            if result.empty:
                print("No data found")
            else:
                display(result)

        elif choice == "2":

            try:
                start = pd.to_datetime(
                    input("Enter start date (YYYY-MM-DD): ")
                )

                end = pd.to_datetime(
                    input("Enter end date (YYYY-MM-DD): ")
                )

                if start > end:
                    print("Invalid date range")
                    return

                result = self.data[
                    (self.data["Date"] >= start) &
                    (self.data["Date"] <= end)
                ]

                if result.empty:
                    print("No data found")
                else:
                    display(result)

            except ValueError:
                print("Enter valid date")

        else:
            print("Invalid choice")

    # Display dataset
    def display_data(self):
        display(self.data)

    # Dataset summary
    def show_summary(self):

        print("Rows:", self.data.shape[0])
        print("Columns:", self.data.shape[1])

        print("\nMissing Values")
        print(self.data.isnull().sum())

        print("\nStatistical Summary")
        display(self.data.describe())

In [ ]:
obj = RetailAnalyzer()

file_name = "retail_sales.csv"

loaded = obj.load_data(file_name)

if loaded:
    obj.calculate_metrics()
    obj.filter_data()
    obj.aggregate_data()

Data loaded successfully
Total Sales: 484100
Average Sales: 40341.67
Most Popular Product: Mouse
Growth Percentage: -94.0 %
1. Filter by Category
2. Filter by Date Range


In [ ]:
if loaded:
    obj.calculate_metrics()

In [ ]:
if loaded:
    obj.filter_data()
    obj.aggregate_data()

In [ ]:
if loaded:

    category_sales = obj.data.groupby(
        "Category"
    )["Total Sales"].sum()

    plt.figure(figsize=(8, 5))

    sns.barplot(
        x=category_sales.index,
        y=category_sales.values
    )

    plt.title("Total Sales by Product Category")
    plt.xlabel("Product Category")
    plt.ylabel("Total Sales")
    plt.legend(["Category Sales"])
    plt.tight_layout()
    plt.show()

In [ ]:
if loaded:

    daily_sales = obj.data.groupby(
        "Date"
    )["Total Sales"].sum().sort_index()

    plt.figure(figsize=(9, 5))

    plt.plot(
        daily_sales.index,
        daily_sales.values,
        marker="o",
        label="Daily Sales"
    )

    plt.title("Sales Trend Over Time")
    plt.xlabel("Date")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if loaded:

    corr = obj.data[
        ["Price", "Quantity Sold"]
    ].corr()

    plt.figure(figsize=(6, 4))

    sns.heatmap(
        corr,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Price and Quantity Sold Correlation")
    plt.tight_layout()
    plt.show()

In [ ]:
if loaded:
    obj.show_summary()

In [ ]:
if loaded:

    while True:

        print("\n===== RETAIL SALES DATA ANALYZER =====")
        print("1. Display Data")
        print("2. Calculate Metrics")
        print("3. Aggregate Data")
        print("4. Filter Data")
        print("5. Dataset Summary")
        print("6. Exit")

        choice = input("Enter your choice: ")

        if choice == "1":
            obj.display_data()

        elif choice == "2":
            obj.calculate_metrics()

        elif choice == "3":
            obj.aggregate_data()

        elif choice == "4":
            obj.filter_data()

        elif choice == "5":
            obj.show_summary()

        elif choice == "6":
            print("Thank you!")
            break

        else:
            print("Invalid choice")